# Connect to Postgres database and setup tables

Now that we have a database up and running we want to be able to connect to it from a python/jupyter environment.

Below is an example of using GeoPandas to load and read data from the database. See [https://postgis.gishub.org/chapters/geopandas.html](https://postgis.gishub.org/chapters/geopandas.html)

## Setup

In [ ]:
from sqlalchemy import URL, create_engine
import geopandas as gpd
import dotenv
import os
import morpc

Load in environmental variables. These include the username and password to acces the database.

In [ ]:
dotenv.load_dotenv(".env")

Create the URL for the location of the database. 

In [ ]:
url = URL.create(
    "postgresql",
    username=os.environ.get('POSTGRES_USER'),
    password=os.environ.get('POSTGRES_PASSWORD'),
    host='localhost', # This is for the locally hosted database. Change to server IP if remote.
    port=5432,
    database='morpc-postgis'
)

Create the engine

In [ ]:
engine = create_engine(url, echo=True)

## Load data

Get some data to load

In [ ]:
gdf = morpc.census.geos.fetch_geos_from_scale_scope('region15', 'place')

Change to column names to lower case. This is the default for postgres tables, and otherwise will it will require more complicated sql strings

In [ ]:
gdf = gdf.reset_index()
gdf.columns = [column.lower() for column in gdf.columns]

View the data

In [ ]:
gdf.crs

Load the data in to the table.

> NOTE: We are using underscores for the table names. This is also the default in postgres tables and will help with constructing sql strings.

GeoPandas allows us to handle if the table already exists. In this case we will replace the table if it exists. Other options are 'fail' and 'append'

In [ ]:
gdf.to_postgis('morpc_geos_region15_place', engine, if_exists='replace') 

Check for the table

In [ ]:
from sqlalchemy import inspect

inspect(engine).get_table_names()

Check the columns in the table

In [ ]:
inspect(engine).get_columns('morpc_geos_region15_place')

## Read the data from the database

In [ ]:
sql = 'SELECT * FROM morpc_geos_region15_county'
gdf = gpd.read_postgis(sql, engine, geom_col='geometry')

In [ ]:
gdf

In [ ]:
gdf.plot()

In [ ]:
gdf.crs

## Query the data with SQL strings

In [ ]:
sql = "SELECT name, geometry from morpc_geos_region15_county WHERE name = 'Franklin County'"
gdf = gpd.read_postgis(sql, engine, geom_col='geometry')

In [ ]:
gdf.plot()

In [ ]:
sql = """
SELECT
  addresses.fulladdress AS address,
  addresses.geometry AS geometry,
  places.name AS placename
FROM morpc_geos_region15_place AS places
JOIN morpc_addresspoints AS addresses
ON ST_Contains(places.geometry, addresses.geometry)
WHERE places.name = 'Columbus city'
"""

In [ ]:
gdf = gpd.GeoDataFrame.from_postgis(sql, engine, geom_col='geometry')

In [ ]:
gdf.plot(markersize=0.1)